# Train Model: 
This notebook trains a TomoGNN for slice-wise detection using PyTorch Lightning. It covers configuration, dataset loading, model construction, training, and checkpoint logging.

## Overview
- Configure training parameters and paths.
- Load tomogram dataset via `data.MrcDataModule`/helpers.
- Build the DETR-based model with `modules.DetrModel`.
- Train with PyTorch Lightning `Trainer` and log to TensorBoard.
- Save checkpoints and evaluate validation metrics.

## prepare config

In [ ]:
# config for stage1 training for ribosome detection
configs = {
    "seed": 1509,
    "model": {
        "name": "conditional_detr", 
        "task":"detection",
        # make sure to use the correct path for the pretrained model
        "load": "<pathto>/release_model/pretrained_models/conditionaldetr.ckpt",
        "output_dim": 2,
        "feature_dim": 256,
        "additional_input_dim": 262,
        "layer_type": "TransformerConv",
        "args": {
            "num_labels": 1,
            "num_queries": 300
        },
        "pick_num":8,
        "class_weight": [1, 0.6],
        "dropout":True, 
        # if the mask is not used, set to "stage 1"
        "stage":"stage 1 mask", 
        "mask_in_channels":1
        
    },
    "training": {
        "name": "conditional_fold1_ribo",
        "logger_path": "./logs",
        "lr_backbone": 1e-5,
        "lr": 1e-4,
        "lr_detr":1e-4,
        "scheduler_step": 2,
        "epochs": 3,
        "gradient_clip_val":None,
    },
    "data": {
        "mrc":True,
        "annotation_path_train": [
            "<pathto>/release_model/data_example/ribosome_label.pkl"
        ],
        "annotation_path_val": [
            "<pathto>/release_model/data_example/ribosome_label.pkl"
        ],
        "transform":"default", 
        "norm":"hist", 
        "require_mask":True,
        "num":1, 
        "gap":1, 
        "map_class":{"ribosome":0 },
        "length_for_average":3
    }
}

## Configuration & Paths
Define and review `configs`: training epochs, accumulation, logging frequency, normalization, and dataset settings. Confirm experiment path, logger directory, and checkpoint naming.

In [2]:
import sys 
sys.path.append("../src/")
import pytorch_lightning as L
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
import importlib 
import data
importlib.reload(data)

import utils

/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
L.seed_everything(configs["seed"])
# logger_path = configs["training"]["logger_path"]
# monitor = "total_validate_loss"
ds = data.get_stage12_dataset_mrc(configs)

Seed set to 1509


reading dataset: <pathto>/release_model/data_example/ribosome_label.pkl
dataset map classes:  {'ribosome': 0}
dataset length: 422
using 1 slices for one graph
reading dataset: <pathto>/release_model/data_example/ribosome_label.pkl
dataset map classes:  {'ribosome': 0}
dataset length: 422
using 1 slices for one graph


## Model Construction
Build the DETR-based model via `utils.getModel(configs)` or `modules.DetrModel`. Ensure stage-specific parameters are set (e.g., mask channels, learning rates).

In [4]:
model = utils.getModel(configs)
trainer = Trainer(
    devices=[1],
    accelerator="gpu",
    max_epochs=configs["training"]["epochs"],
    gradient_clip_val=configs["training"]["gradient_clip_val"],
    accumulate_grad_batches=8,
    log_every_n_steps=5,
)

model at stage  stage 1 mask
model with output classes 2
model receiving class weights tensor([1.0000, 0.6000])
using consistency regularization coef 0.5


/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
/home/feity/cryoem/notebooks/../src/utils.py:1356: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will

incompatible parameters ['model.class_labels_classifier.weight', 'model.class_labels_classifier.bias', 'additional_input_layer.layer1.weight', 'additional_input_layer.layer1.bias', 'additional_input_layer.layer2.weight', 'additional_input_layer.layer2.bias', 'gnn.conv1.lin_l.weight', 'gnn.conv1.lin_l.bias', 'gnn.conv1.lin_r.weight', 'gnn.conv2.lin_l.weight', 'gnn.conv2.lin_l.bias', 'gnn.conv2.lin_r.weight', 'gnn.conv3.lin_l.weight', 'gnn.conv3.lin_l.bias', 'gnn.conv3.lin_r.weight', 'gnn.cls_head.0.weight', 'gnn.cls_head.0.bias', 'gnn.cls_head.2.weight', 'gnn.cls_head.2.bias', 'gnn.box_head.0.weight', 'gnn.box_head.0.bias', 'gnn.box_head.2.weight', 'gnn.box_head.2.bias', 'gnn.edge_head.0.weight', 'gnn.edge_head.0.bias', 'gnn.edge_head.2.weight', 'gnn.edge_head.2.bias', 'gnn.edge_head.4.weight', 'cri.weight']
finish loading parameters


In [5]:
trainer.fit(model, ds)

You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name       | Type                              | Params | Mode 
-------------------------------------------------------------------------
0 | model      | ConditionalDetrForObjectDetection | 43.4 M | train
1 | acc        | MulticlassAccuracy                | 0      | train
2 | auroc      | MulticlassAUROC                   | 0      | train
3 | mask_auroc | BinaryAUROC                       | 0      | train
4 | gnn        | GCN                               | 1.2 M  | train
5 | box_loss   | CompositeSegBBoxLoss              | 0      | train
6 | cri        | CrossEntropyLoss    

/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:78: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 1. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Epoch 2: 100%|██████████| 422/422 [07:32<00:00,  0.93it/s, v_num=36, training_loss=5.490, validation_loss=2.870, total_validate_mask_auroc=0.751, total_validate_loss_ce=0.674, total_validate_loss_bbox=0.0264, total_validate_loss=2.870, total_train_mask_auroc=0.781, total_train_loss=12.70]

`Trainer.fit` stopped: `max_epochs=3` reached.


Epoch 2: 100%|██████████| 422/422 [07:35<00:00,  0.93it/s, v_num=36, training_loss=5.490, validation_loss=2.870, total_validate_mask_auroc=0.751, total_validate_loss_ce=0.674, total_validate_loss_bbox=0.0264, total_validate_loss=2.870, total_train_mask_auroc=0.781, total_train_loss=12.70]


In [6]:
# config for stage1 training for ribosome detection
configs = {
    "seed": 1509,
    "model": {
        "name": "conditional_detr", 
        "task":"detection",
        "load": "<pathto>/release_model/pretrained_models/conditionaldetr.ckpt",
        "output_dim": 2,
        "feature_dim": 256,
        "additional_input_dim": 262,
        "layer_type": "TransformerConv",
        "args": {
            "num_labels": 1,
            "num_queries": 300
        },
        "pick_num":4,
        "class_weight": [1, 0.6],
        "dropout":True, 
        "stage":"stage 1 + 2", 
        "mask_in_channels":1
        
    },
    "training": {
        "name": "conditional_fold1_ribo",
        "logger_path": "./logs",
        "lr_backbone": 0.0,
        "lr": 1e-4,
        "lr_detr":0.0,
        "scheduler_step": 2,
        "epochs": 3,
        "gradient_clip_val":None,
    },
    "data": {
        "mrc":True,
        "annotation_path_train": [
            "<pathto>/release_model/data_example/ribosome_label.pkl"
        ],
        "annotation_path_val": [
            "<pathto>/release_model/data_example/ribosome_label.pkl"
        ],
        "transform":"default", 
        "norm":"hist", 
        "require_mask":False,
        "num":15, 
        "gap":1, 
        "map_class":{"ribosome":0 },
        "length_for_average":3
    }
}

In [7]:
trainer = Trainer(
    devices=[1],
    accelerator="gpu",
    max_epochs=configs["training"]["epochs"],
    # val_check_interval=1000,
    accumulate_grad_batches=8,
    log_every_n_steps=5,
        )
ds = data.get_stage12_dataset_mrc(configs)
model.stage = "stage 1 + 2"
# model = utils.getModel(configs)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


reading dataset: <pathto>/release_model/data_example/ribosome_label.pkl
dataset map classes:  {'ribosome': 0}
dataset length: 422
using 15 slices for one graph
reading dataset: <pathto>/release_model/data_example/ribosome_label.pkl
dataset map classes:  {'ribosome': 0}
dataset length: 422
using 15 slices for one graph


In [8]:
trainer.fit(model, ds)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name       | Type                              | Params | Mode 
-------------------------------------------------------------------------
0 | model      | ConditionalDetrForObjectDetection | 43.4 M | train
1 | acc        | MulticlassAccuracy                | 0      | train
2 | auroc      | MulticlassAUROC                   | 0      | train
3 | mask_auroc | BinaryAUROC                       | 0      | train
4 | gnn        | GCN                               | 1.2 M  | train
5 | box_loss   | CompositeSegBBoxLoss              | 0      | train
6 | cri        | CrossEntropyLoss                  | 0      | train
7 | edge_cri   | BCEWithLogitsLoss                 | 0      | train
8 | kv         | KLDivLoss                         | 0      | train
9 | mask_head  | VitForMask                        | 21.8 M | train
-------------------------------------------------------------------------
66.1 M    Trainable params
222 K     Non-trainable para

Sanity Checking DataLoader 0:  50%|█████     | 1/2 [00:03<00:03,  0.29it/s]

/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:78: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 15. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Epoch 2: 100%|██████████| 422/422 [18:24<00:00,  0.38it/s, v_num=37, training_loss=2.500, validation_loss=0.441, total_validate_auroc=0.812, total_validate_loss_boxes=8.7e-5, total_validate_loss=0.441, total_train_auroc=0.587, total_train_loss_boxes=0.000117, total_train_loss=2.720] 

`Trainer.fit` stopped: `max_epochs=3` reached.


Epoch 2: 100%|██████████| 422/422 [18:27<00:00,  0.38it/s, v_num=37, training_loss=2.500, validation_loss=0.441, total_validate_auroc=0.812, total_validate_loss_boxes=8.7e-5, total_validate_loss=0.441, total_train_auroc=0.587, total_train_loss_boxes=0.000117, total_train_loss=2.720]


## Combine Stage 1 and Stage 2
This section runs sequential training: first Stage 1 (often using slice masks to learn robust features), then Stage 2 (detection/fine-tuning without masks). The best checkpoint from Stage 1 is loaded before Stage 2.

What the script does
- Reads `<path>/config.json` and determines the stage flow from `model.stage` and `training.epochs`.
- Stage 1: trains with `data.require_mask=True` and typically smaller `data.num` for focused learning; saves checkpoints and logs to TensorBoard.
- Loads the best Stage 1 checkpoint (by validation metric).
- Stage 2: sets `data.require_mask=False`, increases `data.num` and continues training for detection; saves checkpoints and logs.

Inputs & assumptions
- Annotation pickle paths listed in `data.annotation_path_train` and `data.annotation_path_val`.
- `data.map_class` maps class names (e.g., `ribosome`) to integer ids.
- GPU devices are set via `-d` and strategy via `--strategy` if needed.

Outputs
- Checkpoints written under the experiment folder (e.g., `stage1/`, `stage2/` or directly under `<path>` depending on the script).
- TensorBoard logs under `training.logger_path` with the run `training.name`.

Usage
- Prepare `<path>/config.json` with stage-specific data settings (e.g., mask vs no-mask) and `training.epochs` (single int or `[stage1_epochs, stage2_epochs]`).
- Launch the two-stage script:

In [ ]:
# now combine the two stages
from sympy import true


configs = {
    "seed": 1509,
    "model": {
        "name": "conditional_detr", 
        "task":"detection",
        "load": "<pathto>/release_model/pretrained_models/conditionaldetr.ckpt",
        "output_dim": 2,
        "feature_dim": 256,
        "additional_input_dim": 262,
        "layer_type": "TransformerConv",
        "args": {
            "num_labels": 1,
            "num_queries": 300
        },
        "class_weight": [1, 0.6],
        "dropout":True
    },
    "training": {
        "name": "conditional_fold1_ribo",
        "logger_path": "/data/transformer_project/transformer_model/train_data/training_final/logs",
        "epochs":[15, 12]
    },
    "data": {
        "mrc":True,
        "annotation_path_train": [
            "<pathto>/release_model/data_example/ribosome_label.pkl"
        ],
        "annotation_path_val": [
            "<pathto>/release_model/data_example/ribosome_label.pkl"
        ],
        "gap":1, 
        "map_class":{"ribosome":0 },
        "length_for_average":3
    }
}

In [ ]:
# ../examples/ the folder where the config file is located, -d 1 means use gpu 1
!python ../scripts/train_full.py -p ../examples/ -d 1

Example command
```bash
python ../scripts/train_full.py -p ../examples/ -d 1
```